# Burton Equation Implementation / Burton 방정식 구현
## "An Empirical Relationship between Interplanetary Conditions and Dst" (Burton, McPherron & Russell, 1975)

이 노트북은 Burton et al. (1975)의 경험적 Dst 예측 방정식을 처음부터 구현하고, 다양한 태양풍 시나리오에 대해 시뮬레이션합니다.

This notebook implements the Burton et al. (1975) empirical Dst prediction equation from scratch and simulates it for various solar wind scenarios.

### 구현 내용 / Contents
1. **Part 1**: Burton equation 핵심 구현 (from scratch) / Core equation implementation
2. **Part 2**: Half-wave rectifier 시각화 / Half-wave rectifier visualization
3. **Part 3**: 단일 폭풍 시뮬레이션 / Single storm simulation
4. **Part 4**: 매개변수 감도 분석 / Parameter sensitivity analysis
5. **Part 5**: 감쇠 시간 분석 / Decay time analysis
6. **Part 6**: 논문의 폭풍 재현 (Feb 23-24, 1967 유사) / Paper storm reproduction
7. **Part 7**: 극한 폭풍 시뮬레이션 / Extreme storm simulation
8. **Part 8**: 현대적 개선과의 비교 / Comparison with modern improvements

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Burton Equation — 핵심 구현 / Core Implementation

Burton equation의 핵심은 에너지 수지 1차 ODE입니다:

The core of the Burton equation is an energy balance first-order ODE:

$$\frac{dDst^*}{dt} = Q(E_y) - \frac{Dst^*}{\tau}$$

여기서:
- $Dst^* = Dst - b\sqrt{P_{dyn}} + c$ (pressure-corrected Dst)
- $Q(E_y)$: half-wave rectifier 주입 함수 / injection function
- $\tau \approx 7.7$ hr: ring current 감쇠 시간 / decay time constant

Where:
- $Dst^* = Dst - b\sqrt{P_{dyn}} + c$ (pressure-corrected Dst)
- $Q(E_y)$: half-wave rectifier injection function
- $\tau \approx 7.7$ hr: ring current decay time constant

In [ ]:
# ============================================================
# Burton Equation: Core Implementation (from scratch, no libraries)
# ============================================================

# Burton (1975) original constants
BURTON_PARAMS = {
    "a": 3.6e-5,       # Decay rate [1/s] → τ = 1/a ≈ 7.7 hr
    "b": 15.8,          # Pressure correction coefficient [nT/nPa^(1/2)]
    "c": 20.0,          # Quiet-time offset [nT]
    "d": -1.5e-3,       # Injection rate coefficient [nT/s per mV/m]
    "Ec": 0.5,          # Threshold electric field [mV/m]
}


def dawn_dusk_efield(v_sw: np.ndarray, bz: np.ndarray) -> np.ndarray:
    """Compute dawn-to-dusk interplanetary electric field.

    Args:
        v_sw: Solar wind speed [km/s].
        bz: IMF Bz component [nT].

    Returns:
        Ey: Dawn-dusk electric field [mV/m].
    """
    # Ey = -Vsw * Bz, with unit conversion:
    # [km/s] * [nT] = [1e3 m/s] * [1e-9 T] = 1e-6 V/m = 1e-3 mV/m
    return -v_sw * bz * 1e-3


def dynamic_pressure(n: np.ndarray, v_sw: np.ndarray) -> np.ndarray:
    """Compute solar wind dynamic pressure.

    Args:
        n: Proton number density [cm^-3].
        v_sw: Solar wind speed [km/s].

    Returns:
        Pdyn: Dynamic pressure [nPa].
    """
    mp = 1.6726e-27  # Proton mass [kg]
    # n [cm^-3] → [m^-3]: * 1e6
    # v [km/s] → [m/s]: * 1e3
    # P = 0.5 * mp * n * v^2 [Pa] → [nPa]: * 1e9
    return 0.5 * mp * (n * 1e6) * (v_sw * 1e3)**2 * 1e9


def injection_function(ey: np.ndarray, d: float, ec: float) -> np.ndarray:
    """Half-wave rectifier injection function Q(Ey).

    Args:
        ey: Dawn-dusk electric field [mV/m].
        d: Injection coefficient [nT/s per mV/m].
        ec: Threshold electric field [mV/m].

    Returns:
        Q: Injection rate [nT/s].
    """
    return np.where(ey > ec, d * (ey - ec), 0.0)


def pressure_correct_dst(dst: np.ndarray, pdyn: np.ndarray,
                         b: float, c: float) -> np.ndarray:
    """Compute pressure-corrected Dst*.

    Args:
        dst: Observed Dst [nT].
        pdyn: Dynamic pressure [nPa].
        b: Pressure coefficient [nT/nPa^(1/2)].
        c: Quiet-time offset [nT].

    Returns:
        Dst*: Pressure-corrected Dst [nT].
    """
    return dst - b * np.sqrt(pdyn) + c


def dst_from_dst_star(dst_star: np.ndarray, pdyn: np.ndarray,
                      b: float, c: float) -> np.ndarray:
    """Recover observed Dst from Dst* and dynamic pressure.

    Args:
        dst_star: Pressure-corrected Dst [nT].
        pdyn: Dynamic pressure [nPa].
        b: Pressure coefficient [nT/nPa^(1/2)].
        c: Quiet-time offset [nT].

    Returns:
        Dst: Observed (total) Dst [nT].
    """
    return dst_star + b * np.sqrt(pdyn) - c


def burton_integrate(ey: np.ndarray, pdyn: np.ndarray,
                     dt: float, dst0: float = 0.0,
                     params: dict | None = None) -> dict:
    """Integrate the Burton equation forward in time using Euler method.

    Args:
        ey: Dawn-dusk electric field time series [mV/m].
        pdyn: Dynamic pressure time series [nPa].
        dt: Time step [seconds].
        dst0: Initial Dst value [nT].
        params: Burton parameters dict (uses defaults if None).

    Returns:
        Dictionary with 'dst', 'dst_star', 'Q', 'decay' arrays.
    """
    if params is None:
        params = BURTON_PARAMS

    a = params["a"]
    b = params["b"]
    c = params["c"]
    d = params["d"]
    ec = params["Ec"]

    n = len(ey)
    dst_star = np.zeros(n)
    dst = np.zeros(n)
    q = injection_function(ey, d, ec)

    # Initial condition
    dst_star[0] = pressure_correct_dst(np.array([dst0]), pdyn[:1], b, c)[0]

    # Forward Euler integration
    for i in range(n - 1):
        ddst_dt = q[i] - a * dst_star[i]
        dst_star[i + 1] = dst_star[i] + ddst_dt * dt

    # Convert back to observed Dst
    dst = dst_from_dst_star(dst_star, pdyn, b, c)
    decay = -a * dst_star

    return {"dst": dst, "dst_star": dst_star, "Q": q, "decay": decay}


print("Burton equation core functions defined.")
print(f"  Decay time τ = {1/BURTON_PARAMS['a']:.0f} s = "
      f"{1/BURTON_PARAMS['a']/3600:.1f} hr")
print(f"  Pressure coefficient b = {BURTON_PARAMS['b']:.1f} nT/nPa^(1/2)")
print(f"  Quiet-time offset c = {BURTON_PARAMS['c']:.0f} nT")
print(f"  Injection coefficient d = {BURTON_PARAMS['d']:.1e} nT/s per mV/m")
print(f"  Threshold Ec = {BURTON_PARAMS['Ec']:.1f} mV/m")

## Part 2: Half-Wave Rectifier 시각화 / Half-Wave Rectifier Visualization

Burton equation의 핵심 비선형 요소는 injection function $Q(E_y)$의 half-wave rectifier 형태입니다.
남향 IMF ($B_z < 0$, 따라서 $E_y > 0$)일 때만 에너지가 주입됩니다.

The key nonlinear element is the half-wave rectifier form of $Q(E_y)$.
Energy injection occurs only when IMF is southward ($B_z < 0$, hence $E_y > 0$).

$$Q(E_y) = \begin{cases} 0 & E_y \leq 0.5 \text{ mV/m} \\ d(E_y - 0.5) & E_y > 0.5 \text{ mV/m} \end{cases}$$

In [ ]:
# ============================================================
# Visualize the half-wave rectifier injection function
# Reproduces the concept of Figures 2 and 3 from the paper
# ============================================================

ey_range = np.linspace(-15, 20, 500)
q_values = injection_function(ey_range, BURTON_PARAMS["d"], BURTON_PARAMS["Ec"])

# Convert to nT/hr for intuitive understanding
q_per_hr = q_values * 3600

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Linear scale (like Figure 2 right panel)
ax = axes[0]
ax.plot(ey_range, q_per_hr, "b-", linewidth=2)
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0.5, color="r", linestyle="--", alpha=0.7, label=f"$E_c$ = {BURTON_PARAMS['Ec']} mV/m")
ax.fill_between(ey_range, q_per_hr, 0, where=(ey_range > 0.5),
                alpha=0.15, color="blue", label="Injection active")
ax.fill_between(ey_range, q_per_hr, 0, where=(ey_range <= 0.5),
                alpha=0.15, color="gray", label="No injection")
ax.set_xlabel("$E_y$ = $-V_{sw}B_z$ [mV/m]")
ax.set_ylabel("Injection rate $Q$ [nT/hr]")
ax.set_title("Half-Wave Rectifier (Linear) / 반파 정류기 (선형)")
ax.legend(fontsize=10)
ax.set_xlim(-15, 20)

# Annotate regions
ax.annotate("Northward IMF\n(북향 IMF)\nNo injection",
            xy=(-7, 0), fontsize=10, ha="center", va="bottom",
            color="gray")
ax.annotate("Southward IMF\n(남향 IMF)\nInjection ∝ $E_y$",
            xy=(12, q_per_hr[ey_range > 11.5][0] / 2), fontsize=10,
            ha="center", color="blue")

# Right: Log-log scale (like Figure 3)
ax = axes[1]
ey_pos = ey_range[ey_range > 0.5]
q_pos = np.abs(q_per_hr[ey_range > 0.5])
ax.loglog(ey_pos, q_pos, "b-", linewidth=2, label="Burton: $Q \\propto E_y$ (slope=1)")

# Show slope=2 line for comparison
q_slope2 = 0.5 * ey_pos**2
ax.loglog(ey_pos, q_slope2, "r--", linewidth=1.5, alpha=0.6,
          label="Hypothetical: $Q \\propto E_y^2$ (slope=2)")
ax.set_xlabel("$E_y$ [mV/m]")
ax.set_ylabel("|Injection rate| [nT/hr]")
ax.set_title("Log-Log Scale (cf. Figure 3) / 로그-로그 스케일")
ax.legend(fontsize=10)
ax.set_xlim(0.5, 20)

plt.tight_layout()
plt.savefig("part2_half_wave_rectifier.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figure 3 비교: Burton은 기울기 1(선형)이 기울기 2(제곱)보다")
print("더 나은 적합을 보인다고 결론지었습니다.")

## Part 3: 단일 폭풍 시뮬레이션 / Single Storm Simulation

이상적인 지자기 폭풍의 3단계(sudden commencement → main phase → recovery)를 합성 태양풍 데이터로 시뮬레이션합니다.

Simulate the three phases of an idealized geomagnetic storm (sudden commencement → main phase → recovery) with synthetic solar wind data.

In [ ]:
# ============================================================
# Idealized geomagnetic storm: SC → main phase → recovery
# ============================================================

dt = 150.0  # Time step [s] = 2.5 min (same as paper's Dst resolution)
total_hours = 48
n_steps = int(total_hours * 3600 / dt)
t_hr = np.arange(n_steps) * dt / 3600  # Time in hours

# --- Synthetic solar wind ---
v_sw = np.full(n_steps, 400.0)     # Baseline 400 km/s
n_sw = np.full(n_steps, 5.0)       # Baseline 5 cm^-3
bz = np.full(n_steps, 2.0)        # Baseline northward +2 nT

# Phase 1 (t=4-6h): Shock arrival — speed & density increase
shock_mask = (t_hr >= 4) & (t_hr < 6)
v_sw[shock_mask] = 600.0
n_sw[shock_mask] = 15.0

# Phase 2 (t=6-8h): Transition — speed stays high, Bz turns southward
trans_mask = (t_hr >= 6) & (t_hr < 8)
v_sw[trans_mask] = 550.0
n_sw[trans_mask] = 10.0
bz[trans_mask] = np.linspace(2, -10, trans_mask.sum())

# Phase 3 (t=8-20h): Main phase — sustained southward IMF
main_mask = (t_hr >= 8) & (t_hr < 20)
v_sw[main_mask] = 500.0
n_sw[main_mask] = 8.0
bz[main_mask] = -10.0

# Phase 4 (t=20-22h): IMF turns northward
turn_mask = (t_hr >= 20) & (t_hr < 22)
v_sw[turn_mask] = 450.0
n_sw[turn_mask] = 6.0
bz[turn_mask] = np.linspace(-10, 3, turn_mask.sum())

# Phase 5 (t=22h+): Recovery — quiet solar wind
rec_mask = t_hr >= 22
v_sw[rec_mask] = 400.0
n_sw[rec_mask] = 5.0
bz[rec_mask] = 2.0

# Compute derived quantities
ey = dawn_dusk_efield(v_sw, bz)
pdyn = dynamic_pressure(n_sw, v_sw)

# Run Burton equation
result = burton_integrate(ey, pdyn, dt, dst0=0.0)

# --- Plot (similar to paper's Figures 4-10 format) ---
fig = plt.figure(figsize=(14, 10))
gs = GridSpec(4, 1, height_ratios=[1, 1, 1, 2], hspace=0.08)

# Panel 1: sqrt(Pdyn)
ax1 = fig.add_subplot(gs[0])
ax1.plot(t_hr, np.sqrt(pdyn), "k-", linewidth=1)
ax1.set_ylabel("$\\sqrt{P_{dyn}}$\n[nPa$^{1/2}$]")
ax1.set_xticklabels([])
ax1.set_title("Idealized Geomagnetic Storm Simulation / 이상적 지자기 폭풍 시뮬레이션")

# Panel 2: Ey
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(t_hr, ey, "k-", linewidth=1)
ax2.axhline(0.5, color="r", linestyle="--", alpha=0.5, linewidth=0.8)
ax2.fill_between(t_hr, ey, 0.5, where=(ey > 0.5), alpha=0.2, color="red")
ax2.set_ylabel("$E_y$\n[mV/m]")
ax2.set_xticklabels([])

# Panel 3: IMF Bz
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(t_hr, bz, "k-", linewidth=1)
ax3.axhline(0, color="gray", linestyle="-", alpha=0.5, linewidth=0.5)
ax3.fill_between(t_hr, bz, 0, where=(bz < 0), alpha=0.2, color="blue",
                 label="Southward ($B_z < 0$)")
ax3.set_ylabel("IMF $B_z$\n[nT]")
ax3.set_xticklabels([])
ax3.legend(fontsize=9, loc="lower right")

# Panel 4: Dst (inverted axis like the paper)
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(t_hr, result["dst"], "b-", linewidth=2, label="Predicted Dst")
ax4.plot(t_hr, result["dst_star"], "r--", linewidth=1.5, alpha=0.7,
         label="$Dst^*$ (pressure-corrected)")
ax4.axhline(0, color="k", linewidth=0.5)
ax4.axhline(-50, color="orange", linestyle=":", alpha=0.5)
ax4.axhline(-100, color="red", linestyle=":", alpha=0.5)
ax4.text(46, -48, "Moderate", fontsize=9, color="orange", va="bottom")
ax4.text(46, -98, "Intense", fontsize=9, color="red", va="bottom")
ax4.set_ylabel("Dst [nT]")
ax4.set_xlabel("Time [hours]")
ax4.legend(fontsize=10, loc="lower right")
ax4.set_xlim(0, total_hours)

# Annotate phases
for ax in [ax1, ax2, ax3, ax4]:
    ax.axvspan(4, 6, alpha=0.05, color="red")
    ax.axvspan(8, 20, alpha=0.05, color="blue")
    ax.axvspan(22, total_hours, alpha=0.05, color="green")

ax1.text(5, ax1.get_ylim()[1]*0.85, "SC", ha="center", fontsize=10,
         color="red", fontweight="bold")
ax1.text(14, ax1.get_ylim()[1]*0.85, "Main Phase", ha="center",
         fontsize=10, color="blue", fontweight="bold")
ax1.text(35, ax1.get_ylim()[1]*0.85, "Recovery", ha="center",
         fontsize=10, color="green", fontweight="bold")

plt.savefig("part3_single_storm.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n최저 Dst: {result['dst'].min():.1f} nT (at t = {t_hr[np.argmin(result['dst'])]:.1f} hr)")
print(f"최저 Dst*: {result['dst_star'].min():.1f} nT")
print(f"SC 첨두: {result['dst'].max():.1f} nT (동압 압축 효과)")

## Part 4: 매개변수 감도 분석 / Parameter Sensitivity Analysis

Burton equation의 4개 핵심 매개변수 ($\tau$, $b$, $d$, $E_c$)를 각각 변화시켜 Dst 예측에 미치는 영향을 분석합니다.

Analyze the impact of varying each of the 4 key parameters ($\tau$, $b$, $d$, $E_c$) on Dst prediction.

In [ ]:
# ============================================================
# Parameter sensitivity analysis
# Use the same synthetic storm from Part 3
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Parameter Sensitivity Analysis / 매개변수 감도 분석", fontsize=14)

# --- (a) Decay time τ ---
ax = axes[0, 0]
for tau_hr in [4, 7.7, 12, 20]:
    params = BURTON_PARAMS.copy()
    params["a"] = 1.0 / (tau_hr * 3600)
    res = burton_integrate(ey, pdyn, dt, dst0=0.0, params=params)
    ax.plot(t_hr, res["dst"], linewidth=1.5,
            label=f"τ = {tau_hr} hr")
ax.set_ylabel("Dst [nT]")
ax.set_title("(a) Decay time τ / 감쇠 시간")
ax.legend(fontsize=9)
ax.set_xlim(0, total_hours)

# --- (b) Pressure coefficient b ---
ax = axes[0, 1]
for b_val in [0, 10, 15.8, 25]:
    params = BURTON_PARAMS.copy()
    params["b"] = b_val
    res = burton_integrate(ey, pdyn, dt, dst0=0.0, params=params)
    ax.plot(t_hr, res["dst"], linewidth=1.5,
            label=f"b = {b_val} nT/nPa$^{{1/2}}$")
ax.set_ylabel("Dst [nT]")
ax.set_title("(b) Pressure coefficient $b$ / 동압 계수")
ax.legend(fontsize=9)
ax.set_xlim(0, total_hours)

# --- (c) Injection coefficient d ---
ax = axes[1, 0]
for d_val in [-0.5e-3, -1.0e-3, -1.5e-3, -2.5e-3]:
    params = BURTON_PARAMS.copy()
    params["d"] = d_val
    res = burton_integrate(ey, pdyn, dt, dst0=0.0, params=params)
    d_hr = d_val * 3600
    ax.plot(t_hr, res["dst"], linewidth=1.5,
            label=f"d = {d_hr:.1f} nT/hr per mV/m")
ax.set_ylabel("Dst [nT]")
ax.set_xlabel("Time [hours]")
ax.set_title("(c) Injection coefficient $d$ / 주입 계수")
ax.legend(fontsize=9)
ax.set_xlim(0, total_hours)

# --- (d) Threshold Ec ---
ax = axes[1, 1]
for ec_val in [0.0, 0.5, 1.0, 2.0]:
    params = BURTON_PARAMS.copy()
    params["Ec"] = ec_val
    res = burton_integrate(ey, pdyn, dt, dst0=0.0, params=params)
    ax.plot(t_hr, res["dst"], linewidth=1.5,
            label=f"$E_c$ = {ec_val} mV/m")
ax.set_ylabel("Dst [nT]")
ax.set_xlabel("Time [hours]")
ax.set_title("(d) Threshold $E_c$ / 임계 전기장")
ax.legend(fontsize=9)
ax.set_xlim(0, total_hours)

plt.tight_layout()
plt.savefig("part4_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

print("관찰 / Observations:")
print("(a) τ가 클수록 → 회복이 느리고 폭풍이 더 깊어짐 (에너지 축적)")
print("(b) b가 클수록 → SC 첨두가 높고 동압 효과가 강함")
print("(c) |d|가 클수록 → 주입이 강해져 폭풍이 깊어짐")
print("(d) Ec가 클수록 → 약한 남향 IMF에서 주입 차단 → 폭풍 약화")

## Part 5: 감쇠 시간 분석 / Decay Time Analysis

Recovery phase에서 ring current 에너지가 지수적으로 감쇠하는 것을 확인합니다.
Burton의 Figure 1을 재현 — 다양한 초기 Dst에서의 감쇠율을 도시합니다.

Verify exponential decay of ring current energy during recovery phase.
Reproduce Burton's Figure 1 concept — plot decay rates from various initial Dst values.

$$Dst^*(t) = Dst^*_0 \cdot e^{-t/\tau}, \quad \tau \approx 7.7 \text{ hr}$$

In [ ]:
# ============================================================
# Decay analysis: reproduce Figure 1 concept
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Recovery curves from different initial Dst ---
ax = axes[0]
t_rec = np.linspace(0, 48, 500)  # Hours
tau = 1 / BURTON_PARAMS["a"] / 3600  # Hours

for dst0_star in [-200, -150, -100, -50, -30]:
    dst_star_t = dst0_star * np.exp(-t_rec / tau)
    ax.plot(t_rec, dst_star_t, linewidth=1.5,
            label=f"$Dst^*_0$ = {dst0_star} nT")

ax.axhline(0, color="k", linewidth=0.5)
ax.axhline(-20, color="gray", linestyle=":", alpha=0.5)
ax.text(46, -22, "$-c$ = -20 nT\n(quiet level)", fontsize=8,
        ha="right", color="gray")
ax.set_xlabel("Time since injection stopped [hours]")
ax.set_ylabel("$Dst^*$ [nT]")
ax.set_title("Recovery Curves / 회복 곡선")
ax.legend(fontsize=9)

# Mark e-folding time
ax.axvline(tau, color="red", linestyle="--", alpha=0.5)
ax.text(tau + 0.5, -10, f"τ = {tau:.1f} hr", color="red", fontsize=10)

# --- Right: Decay rate vs Dst* (like Figure 1) ---
ax = axes[1]
dst_star_range = np.linspace(-200, 0, 100)
decay_rate = -BURTON_PARAMS["a"] * dst_star_range * 3600  # nT/hr (positive = decay toward 0)

# This is -dDst*/dt when Q=0: the rate at which Dst* moves toward zero
ax.plot(dst_star_range, decay_rate, "b-", linewidth=2)
ax.set_xlabel("$Dst^*$ [nT]")
ax.set_ylabel("Decay rate $-dDst^*/dt$ [nT/hr]")
ax.set_title("Decay Rate vs $Dst^*$ (cf. Figure 1) / 감쇠율 vs $Dst^*$")
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)

# Annotate the slope
ax.annotate(f"Slope = $a$ = {BURTON_PARAMS['a']*3600:.4f} hr$^{{-1}}$\n"
            f"= 1/τ = 1/{tau:.1f} hr",
            xy=(-100, -BURTON_PARAMS["a"] * (-100) * 3600),
            xytext=(-170, 20),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=10, color="red")

plt.tight_layout()
plt.savefig("part5_decay_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Quantitative decay analysis
print("감쇠 분석 / Decay Analysis:")
print(f"  τ = {tau:.1f} hours")
print(f"  e-folding: Dst*가 37%로 감소하는 데 {tau:.1f}시간")
print(f"  99% 회복: ~5τ = {5*tau:.1f} hours ≈ {5*tau/24:.1f} days")
print(f"\n  Dst* = -100 nT → 12시간 후: {-100*np.exp(-12/tau):.1f} nT")
print(f"  Dst* = -100 nT → 24시간 후: {-100*np.exp(-24/tau):.1f} nT")
print(f"  Dst* = -100 nT → 48시간 후: {-100*np.exp(-48/tau):.1f} nT")

## Part 6: 논문의 폭풍 재현 / Reproducing a Paper Storm (Feb 23-24, 1967 Style)

논문의 Figure 4 (February 23-24, 1967 폭풍)와 유사한 시나리오를 합성 데이터로 재현합니다.
이 폭풍은 Burton이 "이상적 폭풍"으로 선택한 것으로, SC → main phase → recovery의 모든 특징을 보여줍니다.

Reproduce a scenario similar to Figure 4 (Feb 23-24, 1967 storm) with synthetic data.
This storm was selected by Burton as the "ideal" case showing all SC → main → recovery features.

In [ ]:
# ============================================================
# Reproduce Feb 23-24, 1967 storm scenario (Figure 4)
# Approximate solar wind conditions from the paper's description
# ============================================================

dt = 150.0  # 2.5-min resolution (same as paper)
total_hours = 36
n_steps = int(total_hours * 3600 / dt)
t_hr = np.arange(n_steps) * dt / 3600

# Approximate solar wind based on paper's Figure 4 description:
# - Ey oscillates small positive/negative early (0-8h)
# - Dynamic pressure oscillates with small increases (0-8h)
# - ~12h UT: SC — dynamic pressure spike
# - 13-19h: sustained positive Ey (main phase)
# - 19h+: Ey reverses (recovery)

v_sw = np.full(n_steps, 400.0)
n_sw = np.full(n_steps, 5.0)
bz = np.full(n_steps, 1.0)  # Slight northward

# 0-8h: Small oscillations in Ey and Pdyn
early = t_hr < 8
bz[early] = 2 * np.sin(2 * np.pi * t_hr[early] / 3)  # Oscillating Bz
n_sw[early] = 5 + 3 * np.sin(2 * np.pi * t_hr[early] / 4)
n_sw[n_sw < 2] = 2  # Ensure positive

# 8-12h: Dynamic pressure increases, Ey still small
pre_sc = (t_hr >= 8) & (t_hr < 12)
n_sw[pre_sc] = np.linspace(5, 12, pre_sc.sum())
v_sw[pre_sc] = np.linspace(400, 500, pre_sc.sum())
bz[pre_sc] = 1 + 2 * np.sin(2 * np.pi * t_hr[pre_sc] / 2)

# 12h: SC — sudden dynamic pressure increase
sc_mask = (t_hr >= 12) & (t_hr < 13)
v_sw[sc_mask] = 650
n_sw[sc_mask] = 20
bz[sc_mask] = 2  # Still northward during SC

# 13-19h: Main phase — sustained southward IMF
main = (t_hr >= 13) & (t_hr < 19)
v_sw[main] = 550
n_sw[main] = 10
bz[main] = -8  # Sustained southward

# Small positive Ey spikes during early main phase
main_early = (t_hr >= 13) & (t_hr < 14.5)
bz[main_early] = np.where(
    np.sin(2 * np.pi * t_hr[main_early] / 0.5) > 0, -3, -8
)

# 19-20h: Transition to recovery
trans = (t_hr >= 19) & (t_hr < 20)
bz[trans] = np.linspace(-8, 3, trans.sum())
v_sw[trans] = np.linspace(550, 420, trans.sum())
n_sw[trans] = np.linspace(10, 5, trans.sum())

# 20h+: Recovery — quiet conditions with occasional small positive Ey
rec = t_hr >= 20
v_sw[rec] = 420
n_sw[rec] = 5
bz[rec] = 2 + np.sin(2 * np.pi * t_hr[rec] / 4)

# Compute and run
ey_feb = dawn_dusk_efield(v_sw, bz)
pdyn_feb = dynamic_pressure(n_sw, v_sw)
result_feb = burton_integrate(ey_feb, pdyn_feb, dt, dst0=-10.0)

# --- Plot in paper format ---
fig = plt.figure(figsize=(14, 9))
gs = GridSpec(3, 1, height_ratios=[1, 1, 2], hspace=0.08)

ax1 = fig.add_subplot(gs[0])
ax1.plot(t_hr, np.sqrt(pdyn_feb), "k-", linewidth=0.8)
ax1.set_ylabel("$\\sqrt{P_{dyn}}$\n[nPa$^{1/2}$]")
ax1.set_xticklabels([])
ax1.set_title("Simulated Storm (Feb 23-24, 1967 Style) / 시뮬레이션 폭풍",
              fontsize=13)

ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(t_hr, ey_feb, "k-", linewidth=0.8)
ax2.axhline(0.5, color="r", linestyle="--", alpha=0.4, linewidth=0.7)
ax2.fill_between(t_hr, ey_feb, 0.5, where=(ey_feb > 0.5),
                 alpha=0.15, color="red")
ax2.set_ylabel("$E_y$\n[mV/m]")
ax2.set_xticklabels([])

ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(t_hr, result_feb["dst"], "b-", linewidth=1.5,
         label="Predicted Dst (Burton eq.)")
ax3.plot(t_hr, result_feb["dst_star"], "r--", linewidth=1, alpha=0.6,
         label="$Dst^*$")
ax3.axhline(0, color="k", linewidth=0.5)
ax3.set_ylabel("Dst [nT]")
ax3.set_xlabel("Universal Time [hours from 0000 UT]")
ax3.legend(fontsize=10)
ax3.set_xlim(0, total_hours)

# Annotate
ax3.annotate("SC", xy=(12.5, result_feb["dst"][t_hr > 12][0]),
             fontsize=11, color="red", fontweight="bold",
             xytext=(12.5, result_feb["dst"].max() + 5),
             arrowprops=dict(arrowstyle="->", color="red"))
ax3.annotate("Main Phase\n(남향 IMF)",
             xy=(16, result_feb["dst"][(t_hr > 15.5) & (t_hr < 16.5)].min()),
             fontsize=10, color="blue",
             xytext=(24, -40),
             arrowprops=dict(arrowstyle="->", color="blue"))

plt.savefig("part6_feb1967_storm.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"SC 첨두 Dst: {result_feb['dst'].max():.1f} nT")
print(f"Main phase 최저 Dst: {result_feb['dst'].min():.1f} nT")
print(f"Dst - Dst* 차이 (SC 시): "
      f"{(result_feb['dst'] - result_feb['dst_star'])[(t_hr > 12) & (t_hr < 13)].max():.1f} nT")

## Part 7: 극한 폭풍 시뮬레이션 / Extreme Storm Simulation

논문의 마지막 섹션에서 Burton et al.은 Dst < -300 nT 수준의 극한 폭풍에 필요한 조건을 추정했습니다.
다양한 극한 시나리오 — Carrington급 (1859), March 1989, July 2012 근접통과 — 를 시뮬레이션합니다.

In the final section, Burton et al. estimated conditions needed for Dst < -300 nT extreme storms.
We simulate various extreme scenarios — Carrington-class (1859), March 1989, July 2012 near-miss.

In [ ]:
# ============================================================
# Extreme storm simulations
# ============================================================

dt = 150.0
total_hours = 72
n_steps = int(total_hours * 3600 / dt)
t_hr = np.arange(n_steps) * dt / 3600

scenarios = {
    "Moderate\n(typical CME)": {
        "v": 450, "n": 10, "bz": -8, "dur_hr": 6,
        "desc": "Vsw=450, n=10, Bz=-8, 6hr"
    },
    "Intense\n(Mar 1989-like)": {
        "v": 800, "n": 20, "bz": -25, "dur_hr": 8,
        "desc": "Vsw=800, n=20, Bz=-25, 8hr"
    },
    "Super-storm\n(Jul 2012-like)": {
        "v": 1500, "n": 30, "bz": -30, "dur_hr": 6,
        "desc": "Vsw=1500, n=30, Bz=-30, 6hr"
    },
    "Carrington-class\n(1859-like)": {
        "v": 2000, "n": 50, "bz": -50, "dur_hr": 8,
        "desc": "Vsw=2000, n=50, Bz=-50, 8hr"
    },
}

fig, ax = plt.subplots(figsize=(14, 7))
colors = ["#2196F3", "#FF9800", "#F44336", "#9C27B0"]

for (name, sc), color in zip(scenarios.items(), colors):
    # Build solar wind profile: quiet → storm → recovery
    v = np.full(n_steps, 400.0)
    n_p = np.full(n_steps, 5.0)
    bz_sc = np.full(n_steps, 2.0)

    # Storm window: t=6h to t=6+dur
    storm_start = 6
    storm_end = storm_start + sc["dur_hr"]
    storm = (t_hr >= storm_start) & (t_hr < storm_end)
    v[storm] = sc["v"]
    n_p[storm] = sc["n"]
    bz_sc[storm] = sc["bz"]

    # Gradual transition out
    trans_out = (t_hr >= storm_end) & (t_hr < storm_end + 2)
    v[trans_out] = np.linspace(sc["v"], 400, trans_out.sum())
    n_p[trans_out] = np.linspace(sc["n"], 5, trans_out.sum())
    bz_sc[trans_out] = np.linspace(sc["bz"], 2, trans_out.sum())

    ey_sc = dawn_dusk_efield(v, bz_sc)
    pdyn_sc = dynamic_pressure(n_p, v)
    res = burton_integrate(ey_sc, pdyn_sc, dt, dst0=0.0)

    ax.plot(t_hr, res["dst"], color=color, linewidth=2,
            label=f"{name}: min={res['dst'].min():.0f} nT")

# Storm intensity thresholds
ax.axhline(-50, color="gray", linestyle=":", alpha=0.4)
ax.axhline(-100, color="gray", linestyle=":", alpha=0.4)
ax.axhline(-250, color="gray", linestyle=":", alpha=0.4)
ax.text(70, -48, "Moderate", fontsize=9, color="gray", ha="right")
ax.text(70, -98, "Intense", fontsize=9, color="gray", ha="right")
ax.text(70, -248, "Super-storm", fontsize=9, color="gray", ha="right")

ax.axhline(0, color="k", linewidth=0.5)
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Dst [nT]")
ax.set_title("Extreme Storm Scenarios (Burton Equation) / 극한 폭풍 시나리오")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(0, total_hours)

plt.tight_layout()
plt.savefig("part7_extreme_storms.png", dpi=150, bbox_inches="tight")
plt.show()

# Print summary table
print("\n극한 폭풍 시뮬레이션 결과 / Extreme Storm Simulation Results:")
print(f"{'Scenario':<25} {'Vsw':>6} {'n':>4} {'Bz':>5} {'Dur':>4} {'Ey':>7} {'Min Dst':>9}")
print("-" * 65)
for name, sc in scenarios.items():
    ey_val = -sc["v"] * sc["bz"] * 1e-3
    # Steady-state Dst*
    q_ss = BURTON_PARAMS["d"] * (ey_val - BURTON_PARAMS["Ec"])
    dst_ss = q_ss / BURTON_PARAMS["a"]
    name_short = name.split("\n")[0]
    print(f"{name_short:<25} {sc['v']:>5.0f} {sc['n']:>4.0f} {sc['bz']:>5.0f} "
          f"{sc['dur_hr']:>3.0f}h {ey_val:>6.1f} {dst_ss:>8.0f} nT (ss)")

## Part 8: 현대적 개선과의 비교 / Comparison with Modern Improvements

Burton (1975)의 원본 방정식과 O'Brien & McPherron (2000)의 개선 모델을 비교합니다.
핵심 차이점: O'Brien은 **감쇠 시간 $\tau$를 $E_y$의 함수**로 만들었습니다.

Compare the original Burton (1975) equation with O'Brien & McPherron (2000)'s improvement.
Key difference: O'Brien made **decay time $\tau$ a function of $E_y$**.

### O'Brien & McPherron (2000) 개선 / Improvement:

$$\tau(E_y) = 2.40 \exp\left(\frac{9.74}{4.69 + E_y}\right) \text{ [hours]}$$

$$Q(E_y) = -4.4(E_y - 0.49) \text{ [nT/hr]}$$

강한 주입 시 $\tau$가 짧아지고(빠른 감쇠), 약한 주입 시 $\tau$가 길어집니다(느린 감쇠).

During strong injection, $\tau$ shortens (fast decay); during weak injection, $\tau$ lengthens (slow decay).

In [ ]:
# ============================================================
# O'Brien & McPherron (2000) model implementation
# ============================================================

def obrien_mcpherron_integrate(ey: np.ndarray, pdyn: np.ndarray,
                                dt: float, dst0: float = 0.0) -> dict:
    """Integrate the O'Brien & McPherron (2000) modified Burton equation.

    Key improvement: decay time τ depends on Ey.

    Args:
        ey: Dawn-dusk electric field [mV/m].
        pdyn: Dynamic pressure [nPa].
        dt: Time step [seconds].
        dst0: Initial Dst value [nT].

    Returns:
        Dictionary with 'dst', 'dst_star', 'Q', 'tau' arrays.
    """
    b = 7.26   # O'Brien's pressure coefficient [nT/nPa^(1/2)]
    c = 11.0   # O'Brien's quiet-time offset [nT]

    n = len(ey)
    dst_star = np.zeros(n)
    dst = np.zeros(n)
    tau_arr = np.zeros(n)

    # Initial condition
    dst_star[0] = dst0 - b * np.sqrt(pdyn[0]) + c

    for i in range(n):
        # Variable decay time [hours → seconds]
        ey_eff = max(ey[i], 0.0)
        tau_hr = 2.40 * np.exp(9.74 / (4.69 + ey_eff))
        tau_arr[i] = tau_hr

        # Injection [nT/hr → nT/s]
        if ey[i] > 0.49:
            q_i = -4.4 * (ey[i] - 0.49) / 3600  # nT/s
        else:
            q_i = 0.0

        if i < n - 1:
            a_i = 1.0 / (tau_hr * 3600)  # 1/s
            ddst_dt = q_i - a_i * dst_star[i]
            dst_star[i + 1] = dst_star[i] + ddst_dt * dt

    dst = dst_star + b * np.sqrt(pdyn) - c

    q_arr = np.where(ey > 0.49, -4.4 * (ey - 0.49) / 3600, 0.0)
    return {"dst": dst, "dst_star": dst_star, "Q": q_arr, "tau": tau_arr}


# --- Compare on the idealized storm from Part 3 ---
# Recompute Part 3 storm data
dt = 150.0
total_hours = 48
n_steps = int(total_hours * 3600 / dt)
t_hr = np.arange(n_steps) * dt / 3600

v_sw = np.full(n_steps, 400.0)
n_sw = np.full(n_steps, 5.0)
bz = np.full(n_steps, 2.0)

shock_mask = (t_hr >= 4) & (t_hr < 6)
v_sw[shock_mask] = 600.0; n_sw[shock_mask] = 15.0
trans_mask = (t_hr >= 6) & (t_hr < 8)
v_sw[trans_mask] = 550.0; n_sw[trans_mask] = 10.0
bz[trans_mask] = np.linspace(2, -10, trans_mask.sum())
main_mask = (t_hr >= 8) & (t_hr < 20)
v_sw[main_mask] = 500.0; n_sw[main_mask] = 8.0; bz[main_mask] = -10.0
turn_mask = (t_hr >= 20) & (t_hr < 22)
v_sw[turn_mask] = 450.0; n_sw[turn_mask] = 6.0
bz[turn_mask] = np.linspace(-10, 3, turn_mask.sum())
rec_mask = t_hr >= 22
v_sw[rec_mask] = 400.0; n_sw[rec_mask] = 5.0; bz[rec_mask] = 2.0

ey = dawn_dusk_efield(v_sw, bz)
pdyn = dynamic_pressure(n_sw, v_sw)

res_burton = burton_integrate(ey, pdyn, dt, dst0=0.0)
res_obrien = obrien_mcpherron_integrate(ey, pdyn, dt, dst0=0.0)

# --- Plot comparison ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[2, 1])

ax = axes[0]
ax.plot(t_hr, res_burton["dst"], "b-", linewidth=2,
        label="Burton (1975): τ = 7.7 hr (fixed)")
ax.plot(t_hr, res_obrien["dst"], "r-", linewidth=2,
        label="O'Brien & McPherron (2000): τ = f($E_y$)")
ax.axhline(0, color="k", linewidth=0.5)
ax.axhline(-50, color="gray", linestyle=":", alpha=0.4)
ax.axhline(-100, color="gray", linestyle=":", alpha=0.4)
ax.set_ylabel("Dst [nT]")
ax.set_title("Burton (1975) vs O'Brien & McPherron (2000)")
ax.legend(fontsize=11)
ax.set_xlim(0, total_hours)

# Annotate key differences
ax.annotate("Recovery: O'Brien\nrecovers slower\n(longer τ when Ey≈0)",
            xy=(30, res_obrien["dst"][t_hr > 29.5][0]),
            xytext=(35, -60),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=9, color="red")

ax = axes[1]
ax.plot(t_hr, res_obrien["tau"], "r-", linewidth=1.5)
ax.axhline(7.7, color="b", linestyle="--", linewidth=1,
           label="Burton fixed τ = 7.7 hr")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("τ [hours]")
ax.set_title("O'Brien variable decay time / O'Brien 가변 감쇠 시간")
ax.legend(fontsize=10)
ax.set_xlim(0, total_hours)
ax.set_ylim(0, 20)

plt.tight_layout()
plt.savefig("part8_burton_vs_obrien.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBurton (1975): 최저 Dst = {res_burton['dst'].min():.1f} nT")
print(f"O'Brien (2000): 최저 Dst = {res_obrien['dst'].min():.1f} nT")
print(f"\nO'Brien의 핵심 개선: main phase에서 τ가 짧아지고 (빠른 감쇠),")
print(f"recovery에서 τ가 길어짐 (느린 감쇠) → 더 현실적인 회복 곡선")

## Summary / 요약

| Part | 구현 내용 / Content | 핵심 교훈 / Key Lesson |
|------|-------------------|----------------------|
| 1 | Burton equation 핵심 함수 구현 | 1차 ODE + half-wave rectifier로 폭풍 예측 가능 |
| 2 | Half-wave rectifier 시각화 | 남향 IMF만이 ring current에 에너지 주입 (Dungey 확인) |
| 3 | 이상적 폭풍 시뮬레이션 | SC → main phase → recovery의 완전한 재현 |
| 4 | 매개변수 감도 분석 | τ와 d가 가장 민감, Ec는 약한 폭풍에서만 영향 |
| 5 | 감쇠 시간 분석 | τ ≈ 7.7 hr → 99% 회복에 ~38.5시간 (~1.6일) |
| 6 | Feb 23-24, 1967 스타일 재현 | SC의 동압 효과와 main phase의 ring current 효과 분리 |
| 7 | 극한 폭풍 시나리오 | Carrington급: Burton eq.으로 Dst < -3000 nT 예측 (실제로는 모델 한계) |
| 8 | O'Brien (2000)과 비교 | 가변 τ가 회복 단계의 현실성을 크게 개선 |

### 다음 논문과의 연결 / Connection to Next Papers

- **#12 Cowley (1982)**: Burton의 경험식에 물리적 기반을 제공하는 통합 대류 모델
- **#15 Gonzalez et al. (1994)**: Burton equation 기반 폭풍 분류 체계
- **#33 Camporeale et al. (2019)**: Burton equation을 ML로 대체/보완하는 현대적 접근